In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ValueError: mount failed

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/HOCSAU")

APP_DIR = PROJECT_DIR / "app"
CODE_DIR = PROJECT_DIR / "code"
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

print("PROJECT_DIR:", PROJECT_DIR)
print("PROJECT_DIR exists:", PROJECT_DIR.exists())

print("APP_DIR:", APP_DIR.exists(), APP_DIR)
print("CODE_DIR:", CODE_DIR.exists(), CODE_DIR)
print("DATA_DIR:", DATA_DIR.exists(), DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR.exists(), OUTPUT_DIR)

if PROJECT_DIR.exists():
    print("\nNội dung PROJECT_DIR:")
    print(os.listdir(PROJECT_DIR))
else:
    raise FileNotFoundError(f"Không tìm thấy PROJECT_DIR: {PROJECT_DIR}")

required_dirs = [APP_DIR, CODE_DIR, DATA_DIR, OUTPUT_DIR]

for d in required_dirs:
    if not d.exists():
        raise FileNotFoundError(f"Thiếu thư mục: {d}")

if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

print("\nĐã cấu hình path thành công.")
print("CODE_DIR đã thêm vào sys.path:", CODE_DIR)

# MAIN - PHÂN LOẠI CHÓ/MÈO BẰNG CNN

Notebook chính cho đồ án phân loại ảnh chó/mèo.

Phiên bản này train trên dataset kết hợp:

- Dataset cũ: `/content/drive/MyDrive/Colab Notebooks/HOCSAU/data/catdog`
- Microsoft Cats vs Dogs: `/content/drive/MyDrive/Colab Notebooks/HOCSAU/data/catdog_microsoft`
- Dataset gộp: `/content/drive/MyDrive/Colab Notebooks/HOCSAU/data/dataset_combined_main`
- Output: `/content/drive/MyDrive/Colab Notebooks/HOCSAU/output/main`

Ghi chú vận hành:

- Khi chỉ train lại model: không cần tạo lại dataset gộp.
- Khi muốn tạo lại dataset gộp: bật `BUILD_DATASET = True`.
- Khi muốn train model từ đầu: bật `RESET_TRAINING_OUTPUT = True`.
- Vì Colab RAM khoảng 12GB, pipeline không dùng `cache()`, chỉ dùng `prefetch()`.


In [ ]:
# ============================================================
# CELL 1: IMPORT THƯ VIỆN, KẾT NỐI DRIVE, THIẾT LẬP SEED
# ============================================================

import os
import sys
import json
import time
import shutil
import random
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
    CSVLogger,
    BackupAndRestore
)

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

from google.colab import drive

# Kết nối Google Drive nếu chưa mount
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
else:
    print('Drive đã được mount.')

# Seed cố định để kết quả ổn định hơn
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

# Kiểm tra GPU
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPU:', gpus if gpus else 'Chưa phát hiện GPU')


In [ ]:
# ============================================================
# CELL 2: KHAI BÁO ĐƯỜNG DẪN CHÍNH
# ============================================================

# Thư mục gốc chứa dữ liệu
DATA_ROOT_DIR = PROJECT_DIR / 'data'

# Dataset cũ
OLD_DATASET_DIR = DATA_ROOT_DIR / 'catdog'
OLD_TRAIN_DIR = OLD_DATASET_DIR / 'training_set'
OLD_TEST_DIR = OLD_DATASET_DIR / 'test_set'

# Microsoft Cats vs Dogs
MICROSOFT_DATASET_DIR = DATA_ROOT_DIR / 'catdog_microsoft'

# Dataset gộp dùng để train
COMBINED_DATASET_DIR = DATA_ROOT_DIR / 'dataset_combined_main'
TRAIN_DIR = COMBINED_DATASET_DIR / 'training_set'
TEST_DIR = COMBINED_DATASET_DIR / 'test_set'

# Output model/log/hình ảnh
OUTPUT_DIR = PROJECT_DIR / 'output' / 'main'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Thư mục lưu log riêng của bước gộp dataset
DATASET_BUILD_LOG_DIR = OUTPUT_DIR / 'dataset_build_logs'
DATASET_BUILD_LOG_DIR.mkdir(parents=True, exist_ok=True)

# Model paths
BEST_MODEL_PATH = OUTPUT_DIR / 'best_cnn_cats_dogs_v2.keras'
LATEST_MODEL_PATH = OUTPUT_DIR / 'latest_cnn_cats_dogs_v2.keras'
FINAL_MODEL_PATH = OUTPUT_DIR / 'final_cnn_cats_dogs_v2.keras'

# Log paths
TRAINING_LOG_PATH = OUTPUT_DIR / 'training_log_v2.csv'
TRAINING_INFO_PATH = OUTPUT_DIR / 'training_info_v2.txt'
PLOT_SAVE_PATH = OUTPUT_DIR / 'validation_accuracy_and_loss_v2.png'
BACKUP_DIR = OUTPUT_DIR / 'backup_training_v2'
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT_DIR:', DATA_ROOT_DIR)
print('OLD_DATASET_DIR:', OLD_DATASET_DIR)
print('MICROSOFT_DATASET_DIR:', MICROSOFT_DATASET_DIR)
print('COMBINED_DATASET_DIR:', COMBINED_DATASET_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
# ============================================================
# KIỂM TRA VÀ XÓA ẢNH LỖI CHANNEL
# Mục đích:
# - Xóa ảnh corrupt
# - Xóa ảnh có số channel bất thường
# - Tránh lỗi decode_image khi train
# ============================================================

from PIL import Image
from pathlib import Path

DATASET_DIR = COMBINED_DATASET_DIR

VALID_MODES = ["RGB", "RGBA", "L"]

removed_files = []

image_paths = [
    p for p in DATASET_DIR.rglob("*")
    if p.is_file()
]

print("Tổng file cần kiểm tra:", len(image_paths))

for idx, path in enumerate(image_paths):

    try:
        with Image.open(path) as img:

            if img.mode not in VALID_MODES:

                removed_files.append((str(path), img.mode))

                path.unlink()

    except Exception:

        removed_files.append((str(path), "CORRUPTED"))

        try:
            path.unlink()
        except:
            pass

    if (idx + 1) % 1000 == 0:
        print(f"Đã kiểm tra {idx+1}/{len(image_paths)} ảnh")

print("\nHOÀN TẤT KIỂM TRA ẢNH")
print("=" * 60)

print("Số ảnh lỗi đã xóa:", len(removed_files))

if removed_files:
    print("\nMột số ảnh lỗi:")
    for item in removed_files[:10]:
        print(item)
else:
    print("Không phát hiện ảnh lỗi.")


In [ ]:
# ============================================================
# CELL 3: KIỂM TRA CÁC ĐƯỜNG DẪN
# ============================================================

def check_path(path, name):
    path = Path(path)
    if path.exists():
        print(f'OK   - {name}: {path}')
    else:
        print(f'LỖI  - Không tìm thấy {name}: {path}')

print('KIỂM TRA PATH')
print('=' * 70)
check_path(DATA_ROOT_DIR, 'Thư mục data')
check_path(OLD_DATASET_DIR, 'Dataset cũ catdog')
check_path(OLD_TRAIN_DIR / 'cats', 'Train cats cũ')
check_path(OLD_TRAIN_DIR / 'dogs', 'Train dogs cũ')
check_path(OLD_TEST_DIR / 'cats', 'Test cats cũ')
check_path(OLD_TEST_DIR / 'dogs', 'Test dogs cũ')
check_path(MICROSOFT_DATASET_DIR / 'PetImages' / 'Cat', 'Microsoft Cat')
check_path(MICROSOFT_DATASET_DIR / 'PetImages' / 'Dog', 'Microsoft Dog')
check_path(OUTPUT_DIR, 'Output')


In [ ]:
# ============================================================
# CELL 4: TẠO FILE dataset_builder.py NẾU CHƯA CÓ
# Mục đích:
# - Tách riêng phần gộp dataset khỏi notebook train.
# - Những lần sau chỉ cần import và gọi lại khi cần tạo dataset gộp.
# ============================================================

CODE_DIR = PROJECT_DIR / 'code'
CODE_DIR.mkdir(parents=True, exist_ok=True)
DATASET_BUILDER_PATH = CODE_DIR / 'dataset_builder.py'

builder_code = r'''
"""
dataset_builder.py
Tạo dataset gộp cho bài toán phân loại chó/mèo.

Nguyên tắc:
- Dataset cũ: data/catdog/training_set và data/catdog/test_set
- Dataset Microsoft: data/catdog_microsoft/PetImages/Cat và PetImages/Dog
- Dataset gộp: data/dataset_combined_main
- Giữ nguyên test_set cũ để đánh giá công bằng.
- Chỉ thêm ảnh Microsoft vào training_set.
"""

import os
import shutil
import hashlib
import random
from pathlib import Path

import pandas as pd
from PIL import Image

VALID_IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]


def get_image_paths(folder_path):
    """Lấy danh sách ảnh theo đuôi file hợp lệ trong một thư mục."""
    folder_path = Path(folder_path)
    if not folder_path.exists():
        return []
    return [
        p for p in folder_path.rglob("*")
        if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTENSIONS
    ]


def is_valid_image(image_path):
    """Kiểm tra ảnh có mở được không."""
    try:
        with Image.open(image_path) as img:
            img.verify()
        with Image.open(image_path) as img:
            img.convert("RGB")
        return True
    except Exception:
        return False


def calculate_md5(file_path, chunk_size=8192):
    """Tính MD5 để phát hiện ảnh trùng chính xác theo nội dung file."""
    hash_md5 = hashlib.md5()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            hash_md5.update(chunk)
    return hash_md5.hexdigest()


def count_images(folder_path):
    """Đếm số ảnh hợp lệ theo đuôi file."""
    return len(get_image_paths(folder_path))


def copy_images_to_folder(image_paths, target_dir, prefix, source_name, split_name, class_name):
    """Copy ảnh vào thư mục đích và trả về metadata."""
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    records = []

    for idx, image_path in enumerate(image_paths, start=1):
        image_path = Path(image_path)
        suffix = image_path.suffix.lower()
        new_filename = f"{prefix}_{idx:06d}{suffix}"
        target_path = target_dir / new_filename
        shutil.copy2(str(image_path), str(target_path))

        records.append({
            "source": source_name,
            "split": split_name,
            "class": class_name,
            "original_path": str(image_path),
            "new_path": str(target_path),
            "filename": new_filename
        })

    return records


def create_combined_dataset(
    old_dataset_dir,
    microsoft_dataset_dir,
    combined_dataset_dir,
    output_dir,
    microsoft_images_per_class=5000,
    seed=42,
    rebuild=True
):
    """
    Tạo dataset gộp từ dataset cũ và Microsoft Cats vs Dogs.

    Parameters
    ----------
    old_dataset_dir : str/path
        Thư mục dataset cũ, gồm training_set/cats, training_set/dogs,
        test_set/cats, test_set/dogs.
    microsoft_dataset_dir : str/path
        Thư mục Microsoft dataset, thường gồm PetImages/Cat và PetImages/Dog.
    combined_dataset_dir : str/path
        Thư mục dataset gộp cần tạo.
    output_dir : str/path
        Thư mục lưu metadata/log quá trình gộp.
    microsoft_images_per_class : int
        Số ảnh Microsoft muốn thêm cho mỗi lớp.
    seed : int
        Seed để chọn ảnh ổn định giữa các lần chạy.
    rebuild : bool
        True: xóa dataset gộp cũ và tạo lại từ đầu.
        False: nếu dataset gộp đã tồn tại thì không tạo lại.
    """
    random.seed(seed)

    old_dataset_dir = Path(old_dataset_dir)
    microsoft_dataset_dir = Path(microsoft_dataset_dir)
    combined_dataset_dir = Path(combined_dataset_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Nếu không rebuild và dataset gộp đã có, trả về summary nhanh.
    if (not rebuild) and combined_dataset_dir.exists():
        print("Dataset gộp đã tồn tại và rebuild=False. Bỏ qua bước tạo lại dataset.")
        summary = {
            "combined_dataset_dir": str(combined_dataset_dir),
            "output_dir": str(output_dir),
            "rebuild": False,
        }
        return summary

    old_train_cats_dir = old_dataset_dir / "training_set" / "cats"
    old_train_dogs_dir = old_dataset_dir / "training_set" / "dogs"
    old_test_cats_dir = old_dataset_dir / "test_set" / "cats"
    old_test_dogs_dir = old_dataset_dir / "test_set" / "dogs"

    ms_cats_dir = microsoft_dataset_dir / "PetImages" / "Cat"
    ms_dogs_dir = microsoft_dataset_dir / "PetImages" / "Dog"

    combined_train_cats_dir = combined_dataset_dir / "training_set" / "cats"
    combined_train_dogs_dir = combined_dataset_dir / "training_set" / "dogs"
    combined_test_cats_dir = combined_dataset_dir / "test_set" / "cats"
    combined_test_dogs_dir = combined_dataset_dir / "test_set" / "dogs"

    required_paths = [
        old_train_cats_dir, old_train_dogs_dir,
        old_test_cats_dir, old_test_dogs_dir,
        ms_cats_dir, ms_dogs_dir,
    ]

    for path in required_paths:
        if not path.exists():
            raise FileNotFoundError(f"Không tìm thấy thư mục: {path}")

    if rebuild and combined_dataset_dir.exists():
        print(f"Đang xóa dataset gộp cũ: {combined_dataset_dir}")
        shutil.rmtree(combined_dataset_dir)

    for folder in [
        combined_train_cats_dir, combined_train_dogs_dir,
        combined_test_cats_dir, combined_test_dogs_dir,
    ]:
        folder.mkdir(parents=True, exist_ok=True)

    metadata_records = []

    print("\n[1/5] Copy dataset cũ...")
    metadata_records += copy_images_to_folder(
        get_image_paths(old_train_cats_dir), combined_train_cats_dir,
        "old_train_cats", "old", "train", "cats"
    )
    metadata_records += copy_images_to_folder(
        get_image_paths(old_train_dogs_dir), combined_train_dogs_dir,
        "old_train_dogs", "old", "train", "dogs"
    )
    metadata_records += copy_images_to_folder(
        get_image_paths(old_test_cats_dir), combined_test_cats_dir,
        "old_test_cats", "old", "test", "cats"
    )
    metadata_records += copy_images_to_folder(
        get_image_paths(old_test_dogs_dir), combined_test_dogs_dir,
        "old_test_dogs", "old", "test", "dogs"
    )

    print("\n[2/5] Tính hash test_set cũ để tránh rò rỉ dữ liệu...")
    old_test_paths = get_image_paths(old_test_cats_dir) + get_image_paths(old_test_dogs_dir)
    old_test_hashes = set()
    for idx, path in enumerate(old_test_paths, start=1):
        if idx % 1000 == 0:
            print(f"Đã hash {idx}/{len(old_test_paths)} ảnh test cũ...")
        try:
            old_test_hashes.add(calculate_md5(path))
        except Exception:
            pass

    print("\n[3/5] Kiểm tra ảnh Microsoft: lỗi, trùng test_set, trùng nội bộ...")
    microsoft_records = []
    corrupted_records = []
    duplicate_with_test_records = []

    for class_name, folder in [("cats", ms_cats_dir), ("dogs", ms_dogs_dir)]:
        image_paths = get_image_paths(folder)
        print(f"Kiểm tra Microsoft {class_name}: {len(image_paths)} ảnh")

        for idx, image_path in enumerate(image_paths, start=1):
            if idx % 2000 == 0:
                print(f"  {class_name}: đã kiểm tra {idx}/{len(image_paths)} ảnh...")

            image_path = Path(image_path)

            if not is_valid_image(image_path):
                corrupted_records.append({
                    "class": class_name,
                    "path": str(image_path),
                    "reason": "corrupted"
                })
                continue

            try:
                md5 = calculate_md5(image_path)
            except Exception as e:
                corrupted_records.append({
                    "class": class_name,
                    "path": str(image_path),
                    "reason": f"hash_error: {e}"
                })
                continue

            if md5 in old_test_hashes:
                duplicate_with_test_records.append({
                    "class": class_name,
                    "path": str(image_path),
                    "reason": "duplicate_with_old_test",
                    "md5": md5
                })
                continue

            microsoft_records.append({
                "source": "microsoft",
                "class": class_name,
                "source_path": str(image_path),
                "md5": md5
            })

    microsoft_df = pd.DataFrame(microsoft_records)
    corrupted_df = pd.DataFrame(corrupted_records)
    duplicate_test_df = pd.DataFrame(duplicate_with_test_records)

    before_internal_dup = len(microsoft_df)
    if len(microsoft_df) > 0:
        microsoft_df = microsoft_df.drop_duplicates(subset=["md5"], keep="first").reset_index(drop=True)
    internal_duplicate_count = before_internal_dup - len(microsoft_df)

    microsoft_df.to_csv(output_dir / "clean_microsoft_images.csv", index=False, encoding="utf-8-sig")
    corrupted_df.to_csv(output_dir / "corrupted_microsoft_images.csv", index=False, encoding="utf-8-sig")
    duplicate_test_df.to_csv(output_dir / "duplicate_microsoft_with_test.csv", index=False, encoding="utf-8-sig")

    print("\n[4/5] Chọn ảnh Microsoft cân bằng theo lớp...")
    cats_df = microsoft_df[microsoft_df["class"] == "cats"].copy()
    dogs_df = microsoft_df[microsoft_df["class"] == "dogs"].copy()

    selected_per_class = min(microsoft_images_per_class, len(cats_df), len(dogs_df))
    if selected_per_class == 0:
        raise ValueError("Không đủ ảnh Microsoft sạch để chọn cân bằng cats/dogs.")

    selected_cats_df = cats_df.sample(n=selected_per_class, random_state=seed)
    selected_dogs_df = dogs_df.sample(n=selected_per_class, random_state=seed)

    selected_df = pd.concat([selected_cats_df, selected_dogs_df], ignore_index=True)
    selected_df = selected_df.sample(frac=1, random_state=seed).reset_index(drop=True)
    selected_df.to_csv(output_dir / "selected_microsoft_images.csv", index=False, encoding="utf-8-sig")

    print(f"Số ảnh Microsoft chọn mỗi lớp: {selected_per_class}")

    print("\n[5/5] Copy ảnh Microsoft vào training_set gộp...")
    for class_name, target_dir in [("cats", combined_train_cats_dir), ("dogs", combined_train_dogs_dir)]:
        class_df = selected_df[selected_df["class"] == class_name].copy()
        for idx, row in enumerate(class_df.itertuples(), start=1):
            source_path = Path(row.source_path)
            suffix = source_path.suffix.lower()
            new_filename = f"microsoft_train_{class_name}_{idx:06d}{suffix}"
            target_path = target_dir / new_filename
            shutil.copy2(str(source_path), str(target_path))

            metadata_records.append({
                "source": "microsoft",
                "split": "train",
                "class": class_name,
                "original_path": str(source_path),
                "new_path": str(target_path),
                "filename": new_filename,
                "md5": row.md5
            })

    metadata_df = pd.DataFrame(metadata_records)
    metadata_df.to_csv(output_dir / "combined_dataset_metadata.csv", index=False, encoding="utf-8-sig")

    summary_df = metadata_df.groupby(["source", "split", "class"]).size().reset_index(name="count")
    summary_df.to_csv(output_dir / "combined_dataset_summary.csv", index=False, encoding="utf-8-sig")

    summary_text = f"""
TÓM TẮT TẠO DATASET KẾT HỢP
============================================================
Dataset cũ:
{old_dataset_dir}

Microsoft dataset:
{microsoft_dataset_dir}

Dataset kết hợp:
{combined_dataset_dir}

Số ảnh Microsoft yêu cầu mỗi lớp: {microsoft_images_per_class}
Số ảnh Microsoft thực tế chọn mỗi lớp: {selected_per_class}
Số ảnh Microsoft lỗi/hash lỗi: {len(corrupted_records)}
Số ảnh Microsoft trùng với test_set cũ: {len(duplicate_with_test_records)}
Số ảnh Microsoft trùng nội bộ đã loại: {internal_duplicate_count}

Các file log lưu tại:
{output_dir}
"""

    with open(output_dir / "dataset_builder_summary.txt", "w", encoding="utf-8") as f:
        f.write(summary_text)

    print(summary_text)
    print("Bảng summary:")
    print(summary_df)

    return {
        "combined_dataset_dir": str(combined_dataset_dir),
        "output_dir": str(output_dir),
        "selected_per_class": selected_per_class,
        "metadata_csv": str(output_dir / "combined_dataset_metadata.csv"),
        "summary_csv": str(output_dir / "combined_dataset_summary.csv"),
    }

'''

with open(DATASET_BUILDER_PATH, 'w', encoding='utf-8') as f:
    f.write(builder_code)

print('Đã tạo/cập nhật file dataset_builder.py tại:')
print(DATASET_BUILDER_PATH)


In [ ]:
# ============================================================
# CELL 5: TÙY CHỌN TẠO LẠI DATASET GỘP
# Chỉ bật BUILD_DATASET=True khi cần tạo lại dataset_combined_main.
# Nếu dataset gộp đã có sẵn, để False để tiết kiệm thời gian.
# ============================================================

import sys
if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

from dataset_builder import create_combined_dataset

BUILD_DATASET = False          # Đổi thành True nếu muốn tạo lại dataset gộp
REBUILD_COMBINED_DATASET = True # Chỉ có tác dụng khi BUILD_DATASET=True
MICROSOFT_IMAGES_PER_CLASS = 5000

if BUILD_DATASET:
    summary = create_combined_dataset(
        old_dataset_dir=str(OLD_DATASET_DIR),
        microsoft_dataset_dir=str(MICROSOFT_DATASET_DIR),
        combined_dataset_dir=str(COMBINED_DATASET_DIR),
        output_dir=str(DATASET_BUILD_LOG_DIR),
        microsoft_images_per_class=MICROSOFT_IMAGES_PER_CLASS,
        seed=SEED,
        rebuild=REBUILD_COMBINED_DATASET
    )
    print(summary)
else:
    print('BUILD_DATASET=False: bỏ qua bước tạo lại dataset gộp.')
    print('Notebook sẽ đọc dataset từ:', COMBINED_DATASET_DIR)


In [ ]:
# ============================================================
# CELL 6: ĐẾM ẢNH VÀ EDA NHANH DATASET GỘP
# ============================================================

VALID_IMAGE_EXTENSIONS = ['.jpg', '.jpeg', '.png', '.bmp', '.webp']

def count_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return len([
        p for p in folder.rglob('*')
        if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTENSIONS
    ])

count_df = pd.DataFrame([
    {'split': 'train', 'class': 'cats', 'path': str(TRAIN_DIR / 'cats'), 'count': count_images(TRAIN_DIR / 'cats')},
    {'split': 'train', 'class': 'dogs', 'path': str(TRAIN_DIR / 'dogs'), 'count': count_images(TRAIN_DIR / 'dogs')},
    {'split': 'test', 'class': 'cats', 'path': str(TEST_DIR / 'cats'), 'count': count_images(TEST_DIR / 'cats')},
    {'split': 'test', 'class': 'dogs', 'path': str(TEST_DIR / 'dogs'), 'count': count_images(TEST_DIR / 'dogs')},
])

display(count_df)

# Lưu bảng đếm
count_csv = OUTPUT_DIR / 'dataset_count_v2.csv'
count_df.to_csv(count_csv, index=False, encoding='utf-8-sig')
print('Đã lưu bảng đếm:', count_csv)

# Vẽ biểu đồ phân bố
plt.figure(figsize=(8, 5))
x_labels = count_df['split'] + ' - ' + count_df['class']
plt.bar(x_labels, count_df['count'])
for i, value in enumerate(count_df['count']):
    plt.text(i, value + max(count_df['count']) * 0.01, str(value), ha='center')
plt.title('Số lượng ảnh trong dataset gộp')
plt.xlabel('split - class')
plt.ylabel('số lượng ảnh')
plt.xticks(rotation=20)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plot_path = OUTPUT_DIR / 'dataset_count_v2.png'
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print('Đã lưu biểu đồ:', plot_path)


In [ ]:
# ============================================================
# CELL 7: HIỂN THỊ MỘT SỐ ẢNH MẪU TỪ DATASET GỘP
# ============================================================

def get_image_paths(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return [p for p in folder.rglob('*') if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTENSIONS]

def show_sample_images(folder, title, n=8):
    paths = get_image_paths(folder)
    if len(paths) == 0:
        print('Không có ảnh trong:', folder)
        return
    sample_paths = random.sample(paths, min(n, len(paths)))
    cols = 4
    rows = int(np.ceil(len(sample_paths) / cols))
    plt.figure(figsize=(14, 3.5 * rows))
    for i, path in enumerate(sample_paths):
        img = Image.open(path).convert('RGB')
        ax = plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(Path(folder).name)
        plt.axis('off')
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

show_sample_images(TRAIN_DIR / 'cats', 'Ảnh mẫu train - cats', n=8)
show_sample_images(TRAIN_DIR / 'dogs', 'Ảnh mẫu train - dogs', n=8)


In [ ]:
# ============================================================
# CELL 8: ĐỌC DỮ LIỆU, CHIA VALIDATION, TỐI ƯU PIPELINE
# Lưu ý: không dùng cache() vì RAM Colab khoảng 12GB.
# ============================================================

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
AUTOTUNE = tf.data.AUTOTUNE

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VALIDATION_SPLIT,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=True
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VALIDATION_SPLIT,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=True
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

class_names = train_ds_raw.class_names
print('class_names:', class_names)
print("Ghi chú: nếu class_names = ['cats', 'dogs'] thì cats=0, dogs=1")

# Không cache để tránh đầy RAM; chỉ prefetch để tăng tốc pipeline.
train_ds = train_ds_raw.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds_raw.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds_raw.prefetch(buffer_size=AUTOTUNE)

for images, labels in train_ds.take(1):
    print('Train batch images:', images.shape)
    print('Train batch labels:', labels.shape)
    print('Pixel min/max:', tf.reduce_min(images).numpy(), tf.reduce_max(images).numpy())


In [ ]:
# ============================================================
# CELL 9: DATA AUGMENTATION GIỐNG MÔ HÌNH V2
# ============================================================

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
    ],
    name='data_augmentation'
)

print('Đã tạo data augmentation.')


In [ ]:
# ============================================================
# CELL 10: VISUALIZE ẢNH SAU AUGMENTATION
# ============================================================

for images, labels in train_ds_raw.take(1):
    sample_image = images[0]
    sample_label = int(labels[0].numpy()[0])
    sample_class = class_names[sample_label]
    break

plt.figure(figsize=(12, 12))
for i in range(9):
    aug = data_augmentation(tf.expand_dims(sample_image, axis=0), training=True)
    aug_img = tf.clip_by_value(aug[0], 0, 255).numpy().astype('uint8')
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(aug_img)
    plt.title(sample_class)
    plt.axis('off')
plt.suptitle('Ví dụ ảnh sau khi tăng cường dữ liệu', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
aug_plot_path = OUTPUT_DIR / 'data_augmentation_samples_v2.png'
plt.savefig(aug_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print('Đã lưu hình:', aug_plot_path)


In [ ]:
# ============================================================
# CELL 11: XÂY DỰNG MÔ HÌNH CNN V2
# Kiến trúc chính: Conv2D + BatchNorm + MaxPooling + GAP + Dense + Dropout
# ============================================================

tf.keras.backend.clear_session()

INPUT_SHAPE = IMG_SIZE + (3,)

model = keras.Sequential(
    [
        layers.Input(shape=INPUT_SHAPE),
        data_augmentation,
        layers.Rescaling(1./255),

        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(pool_size=(2, 2)),

        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        layers.Dense(1, activation='sigmoid')
    ],
    name='CNN_Cats_vs_Dogs_Main'
)

model.summary()


In [ ]:
# ============================================================
# CELL 12: COMPILE MÔ HÌNH
# ============================================================

LEARNING_RATE = 3e-4

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc')
    ]
)

print('Đã compile mô hình.')
print('Learning rate:', LEARNING_RATE)


In [ ]:
# ============================================================
# CELL 13: TÙY CHỌN RESET OUTPUT NẾU MUỐN TRAIN TỪ ĐẦU
# Nếu muốn train lại từ epoch 0, đặt RESET_TRAINING_OUTPUT=True rồi chạy cell này.
# Nếu muốn resume, để False.
# ============================================================

RESET_TRAINING_OUTPUT = True

if RESET_TRAINING_OUTPUT:
    files_to_remove = [
        BEST_MODEL_PATH,
        LATEST_MODEL_PATH,
        FINAL_MODEL_PATH,
        TRAINING_LOG_PATH,
        OUTPUT_DIR / 'training_history_v2.pkl',
        OUTPUT_DIR / 'training_history_v2.json',
        TRAINING_INFO_PATH,
        PLOT_SAVE_PATH,
    ]
    folders_to_remove = [BACKUP_DIR]

    for f in files_to_remove:
        f = Path(f)
        if f.exists():
            f.unlink()
            print('Đã xóa:', f)

    for d in folders_to_remove:
        d = Path(d)
        if d.exists():
            shutil.rmtree(d)
            print('Đã xóa thư mục:', d)

    BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    print('Đã reset output. Lần train tiếp theo sẽ bắt đầu từ đầu.')
else:
    print('RESET_TRAINING_OUTPUT=False: giữ nguyên checkpoint/log hiện có.')


In [ ]:
# ============================================================
# CELL 14: CALLBACK, CHECKPOINT, RESUME
# ============================================================

callbacks = [
    BackupAndRestore(
        backup_dir=str(BACKUP_DIR),
        save_freq='epoch'
    ),

    ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),

    ModelCheckpoint(
        filepath=str(LATEST_MODEL_PATH),
        save_best_only=False,
        save_freq='epoch',
        verbose=1
    ),

    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    ),

    CSVLogger(
        filename=str(TRAINING_LOG_PATH),
        append=True
    )
]

print('Callbacks đã sẵn sàng.')
print('Best model:', BEST_MODEL_PATH)
print('Latest model:', LATEST_MODEL_PATH)
print('Training log:', TRAINING_LOG_PATH)


In [ ]:
# ============================================================
# CELL 15: HUẤN LUYỆN HOẶC RESUME MÔ HÌNH
# TARGET_EPOCHS là tổng mốc epoch muốn đạt tới, không phải số epoch train thêm.
# ============================================================

TARGET_EPOCHS = 100

if LATEST_MODEL_PATH.exists():
    print('Phát hiện latest model. Đang load để resume...')
    model = tf.keras.models.load_model(str(LATEST_MODEL_PATH))
else:
    print('Chưa có latest model. Train từ model hiện tại.')

if TRAINING_LOG_PATH.exists():
    log_df = pd.read_csv(TRAINING_LOG_PATH)
    initial_epoch = len(log_df)
else:
    initial_epoch = 0

print('Epoch đã train:', initial_epoch)
print('TARGET_EPOCHS:', TARGET_EPOCHS)

if initial_epoch >= TARGET_EPOCHS:
    print('Mô hình đã đạt hoặc vượt TARGET_EPOCHS. Muốn train thêm thì tăng TARGET_EPOCHS.')
else:
    start_time = time.time()

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=TARGET_EPOCHS,
        initial_epoch=initial_epoch,
        callbacks=callbacks,
        verbose=1
    )

    end_time = time.time()
    training_time_minutes = (end_time - start_time) / 60
    print(f'Hoàn tất huấn luyện phiên hiện tại: {training_time_minutes:.2f} phút')


In [ ]:
# ============================================================
# CELL 16: LƯU HISTORY, FINAL MODEL, TRAINING INFO
# ============================================================

HISTORY_PICKLE_PATH = OUTPUT_DIR / 'training_history_v2.pkl'
HISTORY_JSON_PATH = OUTPUT_DIR / 'training_history_v2.json'

if 'history' in globals():
    with open(HISTORY_PICKLE_PATH, 'wb') as f:
        pickle.dump(history.history, f)

    history_dict = {k: [float(x) for x in v] for k, v in history.history.items()}
    with open(HISTORY_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(history_dict, f, indent=4, ensure_ascii=False)

    print('Đã lưu history.')
else:
    print('Không có biến history mới trong phiên hiện tại.')

model.save(str(FINAL_MODEL_PATH))
print('Đã lưu final model:', FINAL_MODEL_PATH)

if TRAINING_LOG_PATH.exists():
    log_df = pd.read_csv(TRAINING_LOG_PATH)
    total_epochs = len(log_df)
    best_val_acc = log_df['val_accuracy'].max()
    best_val_acc_epoch = log_df['val_accuracy'].idxmax() + 1
    best_val_loss = log_df['val_loss'].min()
    best_val_loss_epoch = log_df['val_loss'].idxmin() + 1
    final_train_acc = log_df['accuracy'].iloc[-1]
    final_val_acc = log_df['val_accuracy'].iloc[-1]
    final_train_loss = log_df['loss'].iloc[-1]
    final_val_loss = log_df['val_loss'].iloc[-1]

    info = f'''THÔNG TIN HUẤN LUYỆN MÔ HÌNH CNN
{'=' * 60}
Dataset train: {TRAIN_DIR}
Dataset test: {TEST_DIR}
IMG_SIZE: {IMG_SIZE}
BATCH_SIZE: {BATCH_SIZE}
Learning rate: {LEARNING_RATE}
Total epochs in log: {total_epochs}

Best validation accuracy: {best_val_acc:.4f} tại epoch {best_val_acc_epoch}
Best validation loss: {best_val_loss:.4f} tại epoch {best_val_loss_epoch}

Final train accuracy: {final_train_acc:.4f}
Final validation accuracy: {final_val_acc:.4f}
Final train loss: {final_train_loss:.4f}
Final validation loss: {final_val_loss:.4f}

Best model: {BEST_MODEL_PATH}
Latest model: {LATEST_MODEL_PATH}
Final model: {FINAL_MODEL_PATH}
Training log: {TRAINING_LOG_PATH}
'''

    with open(TRAINING_INFO_PATH, 'w', encoding='utf-8') as f:
        f.write(info)

    print(info)


In [ ]:
# ============================================================
# CELL 17: VẼ BIỂU ĐỒ ACCURACY VÀ LOSS
# ============================================================

if not TRAINING_LOG_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy training log: {TRAINING_LOG_PATH}')

log_df = pd.read_csv(TRAINING_LOG_PATH)
epochs = range(1, len(log_df) + 1)

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.plot(epochs, log_df['accuracy'], color='blue', linewidth=2, label='train')
plt.plot(epochs, log_df['val_accuracy'], color='orange', linewidth=2, label='val')
plt.title('model accuracy')
plt.xlabel('epoch')
plt.ylabel('accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)

plt.subplot(1, 2, 2)
plt.plot(epochs, log_df['loss'], color='blue', linewidth=2, label='train')
plt.plot(epochs, log_df['val_loss'], color='orange', linewidth=2, label='val')
plt.title('model loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Validation accuracy and loss', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.savefig(PLOT_SAVE_PATH, dpi=300, bbox_inches='tight')
plt.show()
print('Đã lưu biểu đồ:', PLOT_SAVE_PATH)


In [ ]:
# ============================================================
# CELL 18: LOAD BEST MODEL VÀ ĐÁNH GIÁ TRÊN TEST SET
# ============================================================

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy best model: {BEST_MODEL_PATH}')

best_model = tf.keras.models.load_model(str(BEST_MODEL_PATH))

test_results = best_model.evaluate(test_ds, verbose=1, return_dict=True)

print('KẾT QUẢ TEST')
print('=' * 60)
for k, v in test_results.items():
    print(f'{k}: {v:.4f}')

TEST_RESULT_PATH = OUTPUT_DIR / 'test_evaluation_results_v2.txt'
with open(TEST_RESULT_PATH, 'w', encoding='utf-8') as f:
    f.write('KẾT QUẢ ĐÁNH GIÁ TRÊN TEST SET\n')
    f.write('=' * 60 + '\n')
    for k, v in test_results.items():
        f.write(f'{k}: {v:.4f}\n')

print('Đã lưu kết quả test:', TEST_RESULT_PATH)


In [ ]:
# ============================================================
# CELL 19: CONFUSION MATRIX, CLASSIFICATION REPORT, ẢNH ĐÚNG/SAI
# ============================================================

y_true = []
y_prob = []
test_images = []

for images, labels in test_ds:
    probs = best_model.predict(images, verbose=0)
    test_images.append(images.numpy())
    y_true.append(labels.numpy())
    y_prob.append(probs)

test_images = np.concatenate(test_images, axis=0)
y_true = np.concatenate(y_true, axis=0).reshape(-1).astype(int)
y_prob = np.concatenate(y_prob, axis=0).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

num_correct = int(np.sum(y_true == y_pred))
num_wrong = int(np.sum(y_true != y_pred))
print('Số ảnh dự đoán đúng:', num_correct)
print('Số ảnh dự đoán sai:', num_wrong)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
CONFUSION_MATRIX_PATH = OUTPUT_DIR / 'confusion_matrix_v2.png'

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix')
plt.grid(False)
plt.savefig(CONFUSION_MATRIX_PATH, dpi=300, bbox_inches='tight')
plt.show()
print('Đã lưu confusion matrix:', CONFUSION_MATRIX_PATH)

# Classification report
report_text = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report_text)

REPORT_TXT_PATH = OUTPUT_DIR / 'classification_report_v2.txt'
REPORT_CSV_PATH = OUTPUT_DIR / 'classification_report_v2.csv'
with open(REPORT_TXT_PATH, 'w', encoding='utf-8') as f:
    f.write(report_text)
report_df = pd.DataFrame(classification_report(y_true, y_pred, target_names=class_names, digits=4, output_dict=True)).transpose()
report_df.to_csv(REPORT_CSV_PATH, encoding='utf-8-sig')
display(report_df)

# Lưu prediction detail
if hasattr(test_ds_raw, 'file_paths'):
    file_paths = test_ds_raw.file_paths
else:
    file_paths = [None] * len(y_true)

pred_df = pd.DataFrame({
    'file_path': file_paths[:len(y_true)],
    'true_label': y_true,
    'true_class': [class_names[i] for i in y_true],
    'pred_label': y_pred,
    'pred_class': [class_names[i] for i in y_pred],
    'prob_dogs': y_prob,
    'is_correct': y_true == y_pred
})
PRED_DETAIL_PATH = OUTPUT_DIR / 'test_predictions_detail_v2.csv'
pred_df.to_csv(PRED_DETAIL_PATH, index=False, encoding='utf-8-sig')
print('Đã lưu prediction detail:', PRED_DETAIL_PATH)

# Visualize ảnh đúng/sai
def show_prediction_examples(indices, title, save_path, num_images=9):
    if len(indices) == 0:
        print('Không có ảnh cho nhóm:', title)
        return
    selected = np.random.choice(indices, size=min(num_images, len(indices)), replace=False)
    plt.figure(figsize=(12, 12))
    for i, idx in enumerate(selected):
        img = test_images[idx].astype('uint8')
        true_name = class_names[int(y_true[idx])]
        pred_name = class_names[int(y_pred[idx])]
        prob = float(y_prob[idx])
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(img)
        plt.title(f'True: {true_name}\nPred: {pred_name}\nP(dogs): {prob:.2f}', fontsize=10)
        plt.axis('off')
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print('Đã lưu:', save_path)

correct_indices = np.where(y_true == y_pred)[0]
wrong_indices = np.where(y_true != y_pred)[0]
show_prediction_examples(correct_indices, 'Một số ảnh dự đoán đúng', OUTPUT_DIR / 'correct_predictions_v2.png')
show_prediction_examples(wrong_indices, 'Một số ảnh dự đoán sai', OUTPUT_DIR / 'wrong_predictions_v2.png')
